In [4]:
import os
import joblib
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from pathlib import Path

def resolve_repo_root():
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent.parent)
    candidates.append(Path.cwd())
    for candidate in candidates:
        if (candidate / "data" / "processed" / "train.csv").exists():
            return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "processed" / "train.csv").exists():
            return candidate
    return candidates[0] if candidates else Path.cwd()

REPO_ROOT = resolve_repo_root()
TRAIN_CSV = REPO_ROOT / "data" / "processed" / "train.csv"
VAL_CSV = REPO_ROOT / "data" / "processed" / "val.csv"
TEST_CSV = REPO_ROOT / "data" / "processed" / "test.csv"
MODEL_PATH = REPO_ROOT / "models" / "depth5_tree.joblib"

FEATURE_COLS = [
    "num_vars", "num_assertions", "num_uninterpreted_funcs",
    "num_func_applications", "max_func_nesting_depth",
    "ast_node_count", "max_depth", "num_arith_ops", "file_size_bytes"
]
TARGET_COL = "result"

In [5]:
def load_split(path):
    df = pd.read_csv(path)
    return df, df[FEATURE_COLS], df[TARGET_COL]

In [6]:
train_df, X_train, y_train = load_split(TRAIN_CSV)
val_df, X_val, y_val = load_split(VAL_CSV)
test_df, X_test, y_test = load_split(TEST_CSV)

X_cv = pd.concat([X_train, X_val], ignore_index=True)
y_cv = pd.concat([y_train, y_val], ignore_index=True)

final_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
final_clf.fit(X_cv, y_cv)

print(f"Trained on {len(X_cv)} rows (train + val combined)")

Trained on 590 rows (train + val combined)


In [7]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(final_clf, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

Model saved to c:\Users\Dylan\Documents\GitHub\smt-feature-classifier\models\depth5_tree.joblib


In [8]:
test_preds = final_clf.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print(f"Final test accuracy: {test_acc:.3f}\n")
print(classification_report(y_test, test_preds))

labels = sorted(y_cv.unique())
print("Confusion matrix (rows=true, cols=predicted):")
print("Labels order:", labels)
print(confusion_matrix(y_test, test_preds, labels=labels))

Final test accuracy: 0.855

              precision    recall  f1-score   support

         sat       0.87      0.91      0.89        43
     unknown       0.71      0.71      0.71         7
       unsat       0.88      0.79      0.83        19

    accuracy                           0.86        69
   macro avg       0.82      0.80      0.81        69
weighted avg       0.86      0.86      0.85        69

Confusion matrix (rows=true, cols=predicted):
Labels order: ['sat', 'unknown', 'unsat']
[[39  2  2]
 [ 2  5  0]
 [ 4  0 15]]


In [9]:
results_df = test_df.copy()
results_df["predicted"] = test_preds
results_df["correct"] = results_df["result"] == results_df["predicted"]

misclassified = results_df[~results_df["correct"]].copy()
print(f"{len(misclassified)} / {len(results_df)} test formulas misclassified\n")

cols_to_show = ["filepath", "result", "predicted"] + FEATURE_COLS
misclassified[cols_to_show].sort_values("result")

10 / 69 test formulas misclassified



,filepath,result,predicted,num_vars,num_assertions,num_uninterpreted_funcs,num_func_applications,max_func_nesting_depth,ast_node_count,max_depth,num_arith_ops,file_size_bytes
1,20230314-Jaroslav-Bendik-Certora/72658_63104da...,sat,unknown,85,812,8,23702,2,205780,15,59004,42170
8,mathsat/Hash/hash_sat_03_03.smt2,sat,unsat,3,1,3,51,3,252,29,40,1534
12,20230314-Jaroslav-Bendik-Certora/3106_1c933134...,sat,unknown,657,2467,29,46637,2,3283256,42,1086938,315721
17,TwoSquares/smtlib.639533.smt2,sat,unsat,3,3,1,1,1,20,4,7,562
40,wisas/xs_25_45.smt2,unknown,sat,13,1,3,604,1,3165,37,685,14830
58,20230314-Jaroslav-Bendik-Certora/52759_af0c476...,unknown,sat,401,9088,27,11679,2,100047,24,21528,559319
2,mathsat/Wisa/xs-08-20-3-2-4-5.smt2,unsat,sat,13,1,3,164,1,898,68,211,5056
10,wisas/xs_8_18.smt2,unsat,sat,13,1,3,196,1,1091,20,243,5439
11,wisas/xs_15_25.smt2,unsat,sat,13,1,3,364,1,1945,27,425,9248
55,mathsat/Wisa/xs-10-17-5-2-2-5.smt2,unsat,sat,13,1,3,204,1,1094,84,255,6020


In [10]:
print("Misclassifications by true label:")
print(misclassified["result"].value_counts())

print("\nMisclassifications by predicted label:")
print(misclassified["predicted"].value_counts())

Misclassifications by true label:
result
sat        4
unsat      4
unknown    2
Name: count, dtype: int64

Misclassifications by predicted label:
predicted
sat        6
unknown    2
unsat      2
Name: count, dtype: int64


In [12]:
comparison = results_df.groupby("correct")[FEATURE_COLS].median()
comparison.index = ["incorrect", "correct"] if False in comparison.index and True in comparison.index else comparison.index
print("median feature values for correct vs incorrect predictions:")
comparison

median feature values for correct vs incorrect predictions:


,num_vars,num_assertions,num_uninterpreted_funcs,num_func_applications,max_func_nesting_depth,ast_node_count,max_depth,num_arith_ops,file_size_bytes
incorrect,13.0,1.0,3.0,284.0,1.0,1519.5,28.0,340.0,7634.0
correct,13.0,1.0,3.0,457.0,1.0,2597.0,116.0,453.0,11830.0


We observe that correct predictions exhibit significantly higher medians in features measuring the raw size of a formula, meaning the model struggles more on smaller, simpler formulas. Thus, the model is mainly predicting solvability based on formula size, but from this alone it is uncertain whether this is an inherent property of QF_UFLIA solving.  

In [13]:
misclassified["source_dir"] = misclassified["filepath"].apply(lambda p: p.split("/")[0])
print("Misclassified formulas by source directory:")
print(misclassified["source_dir"].value_counts())

results_df["source_dir"] = results_df["filepath"].apply(lambda p: p.split("/")[0])
print("\nAccuracy by source directory (for reference):")
print(results_df.groupby("source_dir")["correct"].mean())

Misclassified formulas by source directory:
source_dir
20230314-Jaroslav-Bendik-Certora    3
mathsat                             3
wisas                               3
TwoSquares                          1
Name: count, dtype: int64

Accuracy by source directory (for reference):
source_dir
20230314-Jaroslav-Bendik-Certora    0.700000
TwoSquares                          0.500000
mathsat                             0.926829
wisas                               0.812500
Name: correct, dtype: float64


In [14]:
print(export_text(final_clf, feature_names=FEATURE_COLS))

|--- num_func_applications <= 13747.00
|   |--- file_size_bytes <= 5980.50
|   |   |--- max_func_nesting_depth <= 2.00
|   |   |   |--- num_arith_ops <= 201.00
|   |   |   |   |--- max_depth <= 4.50
|   |   |   |   |   |--- class: unsat
|   |   |   |   |--- max_depth >  4.50
|   |   |   |   |   |--- class: unsat
|   |   |   |--- num_arith_ops >  201.00
|   |   |   |   |--- file_size_bytes <= 5536.00
|   |   |   |   |   |--- class: sat
|   |   |   |   |--- file_size_bytes >  5536.00
|   |   |   |   |   |--- class: unsat
|   |   |--- max_func_nesting_depth >  2.00
|   |   |   |--- num_uninterpreted_funcs <= 3.50
|   |   |   |   |--- num_vars <= 3.50
|   |   |   |   |   |--- class: unsat
|   |   |   |   |--- num_vars >  3.50
|   |   |   |   |   |--- class: sat
|   |   |   |--- num_uninterpreted_funcs >  3.50
|   |   |   |   |--- class: sat
|   |--- file_size_bytes >  5980.50
|   |   |--- num_vars <= 5.50
|   |   |   |--- max_func_nesting_depth <= 2.00
|   |   |   |   |--- class: unsat
|  